# Keras — Build Any Neural Network in Fewer Lines

---

## What Is Keras?

Keras is a **high-level deep learning API** designed to make building and training neural networks as simple as possible. Originally a standalone library (2015, by François Chollet), it is now fully integrated into TensorFlow as `tf.keras` — and since Keras 3 (2023) it can also run on **JAX** and **PyTorch** backends.

Think of the deep learning ecosystem in layers:

```
┌────────────────────────────────────┐
│          YOUR MODEL CODE           │  ← You write this
├────────────────────────────────────┤
│              KERAS                 │  ← High-level API (layers, losses, callbacks)
├────────────────┬───────────────────┤
│   TensorFlow   │   JAX   │ PyTorch │  ← Backend (compute engine)
└────────────────┴───────────────────┘
```

### Real-World Analogy

Building a neural network in raw TensorFlow/PyTorch is like building a car from individual metal pieces — total control but extremely tedious. Keras is like using **LEGO Technic** — each piece (Dense, Conv2D, LSTM) snaps together cleanly, you can still build any structure, but you don't have to manufacture each bolt yourself.

---

## Why Learn Keras Separately After TensorFlow?

The TensorFlow notebook showed you `tf.keras` — the version bundled with TensorFlow. This notebook goes deeper:
- **Advanced layer types**: Embedding, Conv1D, Conv2D, LSTM, GRU, Attention
- **Regularization techniques**: L1/L2, Dropout, Batch Normalization, Layer Normalization
- **Advanced training**: learning rate schedules, gradient clipping, mixed precision
- **Keras 3 multi-backend**: the same code running on TF, JAX, or PyTorch
- **Pre-built applications**: `keras.applications` (ResNet, EfficientNet, MobileNet, etc.)
- **Transfer learning**: fine-tuning pre-trained models for your task

---

## Prerequisites

- Python fundamentals
- NumPy basics
- The TensorFlow notebook (or equivalent familiarity with neural network concepts)

---

## Table of Contents

1. Installation & Setup
2. The Layer Zoo — Every Important Layer Type
3. Regularization — Prevent Overfitting
4. Advanced Optimizers & Learning Rate Schedules
5. Advanced Training Features
6. Pre-built Applications & Transfer Learning
7. Custom Layers and Loss Functions
8. Keras Tuner — Automated Hyperparameter Search
9. Mini Project — Image Classification with Transfer Learning
10. Common Pitfalls
11. Interview Q&A
12. Resources
13. Summary & What's Next

---

**Official Docs:** https://keras.io/api/  
**GitHub:** https://github.com/keras-team/keras  
**Keras Applications:** https://keras.io/api/applications/  
**YouTube — Deep Learning with Keras (TF Dev Summit):** https://www.youtube.com/watch?v=UYgjEFYCnMs  
**Book by the creator:** https://www.manning.com/books/deep-learning-with-python-second-edition  

## 1. Installation & Setup

```bash
# Keras 3 (multi-backend)
pip install keras

# You also need at least one backend
pip install tensorflow   # TF backend (default)
# OR
pip install jax jaxlib   # JAX backend
# OR
pip install torch        # PyTorch backend
```

Set the backend via environment variable before importing:
```bash
export KERAS_BACKEND=jax      # use JAX
export KERAS_BACKEND=torch    # use PyTorch
export KERAS_BACKEND=tensorflow  # default
```

In [ ]:
# We'll use tensorflow as the backend (most common setup)
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import keras
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits, make_classification
import warnings
warnings.filterwarnings('ignore')

print(f"Keras version:      {keras.__version__}")
print(f"Backend:            {keras.backend.backend()}")
print(f"TensorFlow version: {tf.__version__}")

## 2. The Layer Zoo — Every Important Layer Type

A **layer** takes a tensor in and returns a tensor out. It may have trainable weights (Dense, Conv2D) or no weights (Dropout, Flatten).

```
Category          │ Layers
──────────────────┼──────────────────────────────────────────
Core              │ Dense, Activation, Flatten, Reshape
Convolution       │ Conv1D, Conv2D, Conv3D, DepthwiseConv2D
Pooling           │ MaxPool2D, AveragePool2D, GlobalAvgPool2D
Recurrent         │ LSTM, GRU, SimpleRNN, Bidirectional
Normalization     │ BatchNormalization, LayerNormalization
Regularization    │ Dropout, SpatialDropout2D, GaussianNoise
Attention         │ MultiHeadAttention, Attention
Embedding         │ Embedding (for text/categories)
Merge             │ Add, Concatenate, Average, Maximum
```

In [ ]:
# Explore each major layer type with shape walkthrough

print("=== Dense (Fully Connected) ===")
dense = keras.layers.Dense(64, activation='relu')
out = dense(np.random.randn(8, 10).astype(np.float32))
print(f"Input: (8, 10) → Output: {out.shape}")
print(f"Weights: {dense.kernel.shape}, Biases: {dense.bias.shape}")

print("\n=== Conv2D (Image Convolution) ===")
conv = keras.layers.Conv2D(32, kernel_size=3, activation='relu', padding='same')
out = conv(np.random.randn(4, 28, 28, 1).astype(np.float32))  # 4 grayscale 28x28
print(f"Input: (4,28,28,1) → Output: {out.shape}")

print("\n=== LSTM (Sequence Processing) ===")
lstm = keras.layers.LSTM(64, return_sequences=True)
out = lstm(np.random.randn(4, 20, 8).astype(np.float32))  # 4 sequences, len 20, 8 features
print(f"Input: (4,20,8) → Output (return_seq=True): {out.shape}")

lstm2 = keras.layers.LSTM(64, return_sequences=False)
out = lstm2(np.random.randn(4, 20, 8).astype(np.float32))
print(f"Input: (4,20,8) → Output (return_seq=False): {out.shape}")

print("\n=== Embedding (Text/Category → Dense Vector) ===")
emb = keras.layers.Embedding(input_dim=10000, output_dim=128)  # 10k vocab → 128-dim
out = emb(np.random.randint(0, 10000, (4, 50)))  # 4 sentences, 50 tokens each
print(f"Input: (4,50) integer tokens → Output: {out.shape}")

print("\n=== GlobalAveragePooling2D (CNN → Dense bridge) ===")
gap = keras.layers.GlobalAveragePooling2D()
out = gap(np.random.randn(4, 7, 7, 512).astype(np.float32))  # CNN feature maps
print(f"Input: (4,7,7,512) → Output: {out.shape}")

In [ ]:
# Activation functions — choosing the right one matters!
import numpy as np

x = np.linspace(-4, 4, 200).astype(np.float32)

activations = {
    'ReLU':    keras.activations.relu(x),
    'Sigmoid': keras.activations.sigmoid(x),
    'Tanh':    keras.activations.tanh(x),
    'ELU':     keras.activations.elu(x),
    'GELU':    keras.activations.gelu(x),   # used in Transformers
    'Swish':   keras.activations.swish(x),  # used in EfficientNet
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
use_cases = {
    'ReLU': 'Hidden layers (default)',
    'Sigmoid': 'Binary output layer',
    'Tanh': 'RNNs, normalized features',
    'ELU': 'Avoids dying ReLU, slower',
    'GELU': 'Transformers (GPT, BERT)',
    'Swish': 'EfficientNet, MobileNet'
}

for ax, (name, vals) in zip(axes.flat, activations.items()):
    ax.plot(x, np.array(vals), color='steelblue', lw=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(f'{name}\n({use_cases[name]})', fontsize=9)
    ax.set_ylim(-2, 4)
    ax.grid(alpha=0.3)

plt.suptitle('Activation Functions — Visual Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Regularization — Prevent Overfitting

Overfitting = model memorizes training data but fails on new data. Key regularization techniques:

| Technique | How It Works | When to Use |
|---|---|---|
| **Dropout** | Randomly zeros neurons during training | Most common, works almost everywhere |
| **L2 (weight decay)** | Penalizes large weights in the loss | When features are correlated |
| **L1** | Pushes some weights exactly to zero (sparse) | Feature selection |
| **BatchNorm** | Normalizes activations within a batch | Deep networks (speeds training too) |
| **LayerNorm** | Normalizes across the feature dimension | Transformers / RNNs |
| **Early Stopping** | Stop training when val loss stops improving | Always use |
| **Data Augmentation** | Artificially create more training samples | Small image datasets |

In [ ]:
# Compare: Unregularized vs Regularized model on a small dataset

np.random.seed(42)
X_small = np.random.randn(200, 20).astype(np.float32)
# Only first 3 features matter — rest are noise
y_small = (X_small[:, 0] + X_small[:, 1] - X_small[:, 2] > 0).astype(np.float32)

X_s_tr, X_s_te, y_s_tr, y_s_te = train_test_split(X_small, y_small, test_size=0.3, random_state=42)

def build_model(regularized=False):
    if regularized:
        reg = keras.regularizers.L2(0.01)
        return keras.Sequential([
            keras.layers.Dense(256, activation='relu', kernel_regularizer=reg, input_shape=(20,)),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.5),
            keras.layers.Dense(128, activation='relu', kernel_regularizer=reg),
            keras.layers.Dropout(0.4),
            keras.layers.Dense(1, activation='sigmoid')
        ])
    else:
        return keras.Sequential([
            keras.layers.Dense(256, activation='relu', input_shape=(20,)),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dense(1, activation='sigmoid')
        ])

histories = {}
for name, reg in [('Unregularized', False), ('Regularized', True)]:
    m = build_model(regularized=reg)
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    h = m.fit(X_s_tr, y_s_tr, epochs=100, batch_size=32,
               validation_data=(X_s_te, y_s_te), verbose=0)
    histories[name] = h
    val_acc = max(h.history['val_accuracy'])
    tr_acc  = max(h.history['accuracy'])
    print(f"{name:15s} → Train Acc: {tr_acc:.3f}, Best Val Acc: {val_acc:.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in histories.items():
    axes[0].plot(h.history['accuracy'],     label=f'{name} Train')
    axes[0].plot(h.history['val_accuracy'], label=f'{name} Val', linestyle='--')
    axes[1].plot(h.history['loss'],         label=f'{name} Train')
    axes[1].plot(h.history['val_loss'],     label=f'{name} Val', linestyle='--')

for ax, title in zip(axes, ['Accuracy', 'Loss']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.set_title(title); ax.legend(fontsize=7)

plt.suptitle('Effect of Regularization on a Small Dataset', y=1.02)
plt.tight_layout()
plt.show()

## 4. Advanced Optimizers & Learning Rate Schedules

The learning rate (LR) is the most important hyperparameter. A schedule that changes LR during training almost always outperforms a fixed LR.

**Common schedules:**
- **StepDecay**: cut LR by a factor every N epochs
- **ExponentialDecay**: LR shrinks exponentially
- **CosineDecay**: LR follows a cosine curve (warm restarts possible)
- **Warmup + Decay**: start low, ramp up, then decay (used for Transformers)

In [ ]:
# Visualize different LR schedules

steps = np.arange(0, 1000)

schedules = {
    'Constant (1e-3)': [1e-3] * 1000,

    'ExponentialDecay': [
        keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=1e-3, decay_steps=200, decay_rate=0.5
        )(s).numpy() for s in steps
    ],

    'CosineDecay': [
        keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=1e-3, decay_steps=1000
        )(s).numpy() for s in steps
    ],

    'CosineDecayRestarts': [
        keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=1e-3, first_decay_steps=250
        )(s).numpy() for s in steps
    ],
}

fig, ax = plt.subplots(figsize=(10, 4))
for name, lrs in schedules.items():
    ax.plot(steps, lrs, label=name, lw=2)

ax.set_xlabel('Training Step'); ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules Comparison')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Using a schedule in a model
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=5000,  # number of training steps (batches × epochs)
    alpha=1e-5          # minimum LR at end
)
optimizer = keras.optimizers.Adam(learning_rate=lr_schedule)
print(f"LR at step   0: {float(optimizer.learning_rate(0)):.6f}")
print(f"LR at step 500: {float(optimizer.learning_rate(500)):.6f}")
print(f"LR at step 2500: {float(optimizer.learning_rate(2500)):.6f}")
print(f"LR at step 5000: {float(optimizer.learning_rate(5000)):.6f}")

## 5. Advanced Training Features

In [ ]:
# ---- Gradient Clipping (prevents gradient explosion in RNNs) ----
# Caps gradient norm so no single update is too large

optimizer_clipped = keras.optimizers.Adam(
    learning_rate=0.001,
    clipnorm=1.0    # clip gradients when their norm exceeds 1.0
    # clipvalue=0.5  ← alternative: clip each gradient value individually
)
print("Adam with gradient clipping (clipnorm=1.0) created")

# ---- Mixed Precision Training (faster on modern GPUs) ----
# Uses float16 for computations (2x faster on A100/V100) but float32 for weights
# Uncomment the lines below when running on a GPU:
# keras.mixed_precision.set_global_policy('mixed_float16')
# model.compile(...)
# keras.mixed_precision.set_global_policy('float32')  # reset
print("Mixed precision: set via keras.mixed_precision.set_global_policy('mixed_float16')")

# ---- Class Weights (for imbalanced datasets) ----
np.random.seed(42)
n_neg, n_pos = 900, 100  # 9:1 imbalance
X_imb = np.random.randn(n_neg + n_pos, 10).astype(np.float32)
y_imb = np.array([0] * n_neg + [1] * n_pos, dtype=np.float32)

# Compute class weights
total = n_neg + n_pos
class_weights = {
    0: total / (2 * n_neg),  # 0.556
    1: total / (2 * n_pos)   # 5.0
}
print(f"\nClass weights → 0: {class_weights[0]:.3f}, 1: {class_weights[1]:.3f}")

imb_model = keras.Sequential([
    keras.layers.Dense(32, activation='relu', input_shape=(10,)),
    keras.layers.Dense(1, activation='sigmoid')
])
imb_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
imb_model.fit(X_imb, y_imb, epochs=10, class_weight=class_weights, verbose=0)
print("Model trained with class weights!")

## 6. Pre-built Applications & Transfer Learning

`keras.applications` provides **pre-trained models** (trained on ImageNet — 1.2 million images, 1000 classes). You can use them:

1. **As feature extractors**: freeze all weights, only train a new head
2. **Fine-tuning**: unfreeze some top layers and train them with a very low LR

Available models include: `VGG16`, `ResNet50`, `InceptionV3`, `MobileNetV2`, `EfficientNetB0-B7`, `NASNetMobile`, `DenseNet121`, `ConvNeXtTiny`, etc.

**Why transfer learning?** A model trained on 1.2M images has already learned to detect edges, textures, shapes, and high-level objects. You don't need to relearn all that — you just teach it to recognize YOUR specific categories from just a few hundred images.

In [ ]:
# Load MobileNetV2 (lightweight, fast — good for learning)
base_model = keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),  # images must be resized to this
    include_top=False,           # exclude the original 1000-class head
    weights='imagenet'           # load pre-trained ImageNet weights
)

print(f"MobileNetV2 total layers: {len(base_model.layers)}")
print(f"Base model output shape: {base_model.output_shape}")

# ---- Step 1: Feature Extraction (freeze base) ----
base_model.trainable = False  # freeze ALL layers in the base model

frozen_params  = sum(1 for p in base_model.weights if not p.trainable)
trainable_params = sum(1 for p in base_model.weights if p.trainable)
print(f"Frozen params: {frozen_params}, Trainable: {trainable_params}")

In [ ]:
# Build transfer learning model using Functional API
NUM_CLASSES = 5  # e.g., 5 flower species

inputs = keras.Input(shape=(128, 128, 3))

# Preprocessing: normalize pixels to [-1, 1] as expected by MobileNetV2
x = keras.applications.mobilenet_v2.preprocess_input(inputs)

# Base model (feature extractor)
x = base_model(x, training=False)  # training=False → BatchNorm uses stored stats

# Custom classification head
x = keras.layers.GlobalAveragePooling2D()(x)   # (batch, 7, 7, 1280) → (batch, 1280)
x = keras.layers.Dense(256, activation='relu')(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

transfer_model = keras.Model(inputs, outputs, name='TransferLearningModel')

# Only the new head is trainable
trainable = sum(np.prod(w.shape) for w in transfer_model.trainable_weights)
total     = sum(np.prod(w.shape) for w in transfer_model.weights)
print(f"Trainable parameters: {trainable:,} / {total:,} total ({trainable/total:.1%})")

transfer_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Simulate training (normally you'd load real image data)
dummy_images = np.random.randint(0, 256, (64, 128, 128, 3), dtype=np.uint8).astype(np.float32)
dummy_labels = np.random.randint(0, NUM_CLASSES, 64)

transfer_model.fit(dummy_images, dummy_labels, epochs=3, batch_size=16, verbose=1)

print("\nPhase 1 (feature extraction) complete!")
print("Now unfreeze top layers for fine-tuning...")

# ---- Step 2: Fine-Tuning (unfreeze top layers) ----
base_model.trainable = True

# Freeze all but the last 30 layers of the base model
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Re-compile with much lower LR (important!)
transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # 100x lower than before
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

trainable_ft = sum(np.prod(w.shape) for w in transfer_model.trainable_weights)
print(f"Fine-tune trainable params: {trainable_ft:,} / {total:,} ({trainable_ft/total:.1%})")

## 7. Custom Layers and Loss Functions

When built-in layers aren't enough, Keras makes it easy to write your own.

In [ ]:
# ==================================================
# Custom Layer — implements a new operation
# ==================================================

class ScaledDotProductAttention(keras.layers.Layer):
    """
    Simplified single-head attention layer.
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    Used as the core of Transformer architecture.
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_k = d_model
        self.W_q = keras.layers.Dense(d_model)  # query projection
        self.W_k = keras.layers.Dense(d_model)  # key projection
        self.W_v = keras.layers.Dense(d_model)  # value projection

    def call(self, x):
        Q = self.W_q(x)                            # (batch, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(float(self.d_k))
        weights = tf.nn.softmax(scores, axis=-1)   # attention weights
        return tf.matmul(weights, V)               # weighted sum of values

# Test
attn = ScaledDotProductAttention(d_model=64)
seq_input = tf.random.normal([4, 10, 32])  # (batch=4, seq_len=10, features=32)
attn_out = attn(seq_input)
print(f"Attention input:  {seq_input.shape}")
print(f"Attention output: {attn_out.shape}")

In [ ]:
# ==================================================
# Custom Loss Function — Focal Loss
# (better than BCE for class imbalance)
# ==================================================

class FocalLoss(keras.losses.Loss):
    """
    Focal Loss: down-weights easy examples so the model
    focuses on hard-to-classify samples.
    FL(pt) = -alpha * (1 - pt)^gamma * log(pt)
    
    gamma=0 → regular binary cross-entropy
    gamma=2 → typical Focal Loss (from RetinaNet paper)
    """
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)  # avoid log(0)
        bce    = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t    = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = self.alpha * tf.pow(1.0 - p_t, self.gamma)
        return tf.reduce_mean(focal_weight * bce)

# Use the custom loss
focal_model = keras.Sequential([
    keras.layers.Dense(32, activation='relu', input_shape=(10,)),
    keras.layers.Dense(1, activation='sigmoid')
])
focal_model.compile(
    optimizer='adam',
    loss=FocalLoss(gamma=2.0, alpha=0.25),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

X_focal = np.random.randn(500, 10).astype(np.float32)
y_focal = (np.random.rand(500) < 0.1).astype(np.float32)  # 10% positive
focal_model.fit(X_focal, y_focal, epochs=5, batch_size=64, verbose=1)

## 8. Keras Tuner — Automated Hyperparameter Search

Instead of manually trying different architectures and hyperparameters, `keras_tuner` automates this search.

In [ ]:
# Install if needed: pip install keras-tuner
try:
    import keras_tuner as kt
    HAS_KT = True
except ImportError:
    HAS_KT = False
    print("keras-tuner not installed. Run: pip install keras-tuner")

if HAS_KT:
    # Generate data
    np.random.seed(42)
    X_kt, y_kt = make_classification(n_samples=1000, n_features=20,
                                       n_informative=10, random_state=42)
    X_kt = X_kt.astype(np.float32); y_kt = y_kt.astype(np.float32)
    X_kt_tr, X_kt_te, y_kt_tr, y_kt_te = train_test_split(X_kt, y_kt, test_size=0.2)

    def build_tunable_model(hp):
        """hp is the HyperParameters object — defines search space."""
        model = keras.Sequential()
        model.add(keras.layers.Input(shape=(20,)))

        # Search: how many hidden layers? (1, 2, or 3)
        for i in range(hp.Int('num_layers', 1, 3)):
            # Search: how many units per layer?
            model.add(keras.layers.Dense(
                units=hp.Choice(f'units_{i}', [32, 64, 128, 256]),
                activation='relu'
            ))
            # Search: should we use dropout?
            if hp.Boolean(f'dropout_{i}'):
                model.add(keras.layers.Dropout(
                    rate=hp.Float(f'dropout_rate_{i}', 0.1, 0.5, step=0.1)
                ))

        model.add(keras.layers.Dense(1, activation='sigmoid'))

        # Search: what learning rate?
        model.compile(
            optimizer=keras.optimizers.Adam(
                hp.Float('lr', 1e-4, 1e-2, sampling='log')
            ),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        return model

    # RandomSearch tries random combinations
    tuner = kt.RandomSearch(
        build_tunable_model,
        objective='val_accuracy',
        max_trials=5,          # try 5 combinations
        directory='/tmp/kt_results',
        project_name='demo',
        overwrite=True
    )

    tuner.search(
        X_kt_tr, y_kt_tr,
        epochs=10,
        validation_split=0.2,
        callbacks=[keras.callbacks.EarlyStopping(patience=3)],
        verbose=0
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    print("Best hyperparameters found:")
    print(f"  num_layers: {best_hp.get('num_layers')}")
    print(f"  units_0:    {best_hp.get('units_0')}")
    print(f"  lr:         {best_hp.get('lr'):.5f}")

    best_model = tuner.get_best_models(1)[0]
    _, test_acc = best_model.evaluate(X_kt_te, y_kt_te, verbose=0)
    print(f"Best model test accuracy: {test_acc:.4f}")

## 9. Mini Project — Text Sentiment Classification

### The Problem

Classify movie reviews as **positive** or **negative** using a 1D CNN over word embeddings. This covers:
- `Embedding` layer (words → dense vectors)
- `Conv1D` (detects local patterns in text)
- `GlobalMaxPooling1D` (selects the strongest signal)
- Full Keras training pipeline

In [ ]:
# ==================================================
# STEP 1: Simulate text data
# (use IMDB dataset if internet available:
#  (X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data())
# ==================================================

np.random.seed(42)

VOCAB_SIZE  = 5000   # unique words
MAX_LEN     = 200    # max sequence length (words per review)
N_SAMPLES   = 3000

# Simulate integer-encoded reviews: each entry is a word index
# Positive reviews use higher-index words (e.g., "great", "love", "amazing")
# Negative reviews use lower-index words (e.g., "bad", "awful", "boring")
positive_vocab = np.arange(VOCAB_SIZE//2, VOCAB_SIZE)  # "positive" words
negative_vocab = np.arange(1, VOCAB_SIZE//2)           # "negative" words

def gen_review(label, seq_len):
    if label == 1:
        main = np.random.choice(positive_vocab, size=int(seq_len * 0.7))
        noise = np.random.choice(negative_vocab, size=seq_len - len(main))
    else:
        main = np.random.choice(negative_vocab, size=int(seq_len * 0.7))
        noise = np.random.choice(positive_vocab, size=seq_len - len(main))
    seq = np.concatenate([main, noise])
    np.random.shuffle(seq)
    return seq

labels  = np.random.randint(0, 2, N_SAMPLES)
lengths = np.random.randint(50, MAX_LEN, N_SAMPLES)
reviews = [gen_review(l, ln) for l, ln in zip(labels, lengths)]

# Pad/truncate to MAX_LEN
X_text = keras.utils.pad_sequences(reviews, maxlen=MAX_LEN, padding='post', truncating='post')
y_text = labels.astype(np.float32)

X_txt_tr, X_txt_te, y_txt_tr, y_txt_te = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

print(f"Dataset: {X_text.shape}  (samples, sequence_length)")
print(f"Positive: {(y_text==1).sum()}, Negative: {(y_text==0).sum()}")

In [ ]:
# ==================================================
# STEP 2: Build Text CNN model
# ==================================================

EMBED_DIM = 64  # size of word embedding vectors

text_model = keras.Sequential([
    # Embedding: (batch, seq_len) integer tokens → (batch, seq_len, EMBED_DIM)
    keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),

    # Multiple filter sizes to capture n-gram patterns
    # We'll use separate Conv1D layers and concatenate (Functional API would be cleaner)
    keras.layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    keras.layers.Conv1D(128, kernel_size=5, activation='relu', padding='same'),

    # Global max pooling: take the strongest feature across the sequence
    keras.layers.GlobalMaxPooling1D(),

    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(1, activation='sigmoid')
])

text_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

text_model.summary()

In [ ]:
# ==================================================
# STEP 3: Train and evaluate
# ==================================================

text_history = text_model.fit(
    X_txt_tr, y_txt_tr,
    epochs=20,
    batch_size=64,
    validation_split=0.15,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_auc', patience=5, mode='max',
                                       restore_best_weights=True)
    ],
    verbose=1
)

test_results = text_model.evaluate(X_txt_te, y_txt_te, verbose=0)
print(f"\nTest Loss: {test_results[0]:.4f}")
print(f"Test Acc:  {test_results[1]:.4f}")
print(f"Test AUC:  {test_results[2]:.4f}")

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(text_history.history['auc'], label='Train AUC')
axes[0].plot(text_history.history['val_auc'], label='Val AUC')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].set_title('Sentiment Classifier — AUC'); axes[0].legend()

axes[1].plot(text_history.history['loss'], label='Train')
axes[1].plot(text_history.history['val_loss'], label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Binary Cross-Entropy Loss'); axes[1].legend()

plt.tight_layout()
plt.show()

print("\nTo use on real IMDB data, replace the data generation with:")
print("  (X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)")
print("  X_train = keras.utils.pad_sequences(X_train, maxlen=MAX_LEN)")

## 10. Common Pitfalls

### Pitfall 1: `input_shape` vs `Input` Layer

For Sequential models, specify the input shape on the **first layer** (not separately):
```python
# CORRECT
keras.layers.Dense(64, activation='relu', input_shape=(20,))
# OR use Input layer with Functional API
keras.Input(shape=(20,))
```

### Pitfall 2: Output Activation vs Loss Function Mismatch

A very common mistake:
```python
# WRONG: sigmoid output + crossentropy loss (numeric instability)
keras.layers.Dense(1, activation='sigmoid')
model.compile(loss='binary_crossentropy')   # this actually works...

# MORE STABLE: no activation + from_logits=True
keras.layers.Dense(1)  # raw logits
model.compile(loss=keras.losses.BinaryCrossentropy(from_logits=True))
```

### Pitfall 3: `model.predict()` vs calling model directly

- `model.predict(X)` — for large datasets, batches automatically, returns NumPy arrays
- `model(X)` — eager call, loads everything into memory at once, returns tensors  
Always use `model.predict()` for inference on large data.

### Pitfall 4: Forgetting `trainable=False` Before `model.compile()`

When doing transfer learning, set layer trainability **before** compiling:
```python
base_model.trainable = False  ← must come BEFORE compile
model.compile(...)
```
Changing trainability after compile has no effect until you re-compile.

### Pitfall 5: One-Hot vs Sparse Labels

```python
# y is integer class indices [0, 1, 2, ...] → use sparse_categorical_crossentropy
# y is one-hot [[1,0,0], [0,1,0], ...] → use categorical_crossentropy
```
Always check your label format before choosing the loss.

## 11. Interview Q&A

---

**Q1: What is Keras and how does it relate to TensorFlow?**

> Keras is a high-level deep learning API that provides a clean interface for building, training, and deploying neural networks. Originally a standalone library, it became TensorFlow's official high-level API (`tf.keras`) in TF2. Since Keras 3 (2023), it also supports JAX and PyTorch as backends, making Keras code truly backend-agnostic. The key abstraction: Keras handles the "what" (architecture, training loop, metrics), while the backend handles the "how" (tensor math, gradient computation, hardware acceleration).

---

**Q2: What is Batch Normalization and why does it help?**

> BatchNorm normalizes the activations of each layer within a mini-batch: `(x - mean) / std`, then scales with learnable gamma and shifts with learnable beta. Benefits:
> 1. **Faster training**: activations stay in a useful range, gradients don't vanish/explode
> 2. **Regularization effect**: each sample's normalization depends on the batch, adding noise that acts like regularization
> 3. **Less sensitive to initialization and LR choice**
> 
> During inference, it uses running statistics (exponential moving average of training batch stats) rather than the current batch stats. Important: always pass `training=True/False` correctly in custom models.

---

**Q3: What is the difference between Dropout and BatchNormalization?**

> Both are regularization techniques but work differently:
> - **Dropout**: randomly zeros `p` fraction of neurons each training step — forces redundancy, reduces co-adaptation
> - **BatchNorm**: normalizes activations — speeds training and has mild regularization effect
> 
> They can both be used in the same model, but don't stack Dropout immediately after BatchNorm — they interfere (Dropout adds variance; BatchNorm tries to remove variance). Use: `Conv → BN → ReLU → Dropout`.

---

**Q4: What is transfer learning and when is it useful?**

> Transfer learning uses a model pre-trained on a large dataset as the starting point for a different (usually smaller) task. The pre-trained model has already learned low-level features (edges, textures, shapes for images; grammar, word relationships for text). Transfer learning is most useful when:
> - Your dataset is small (< 10,000 samples)
> - The source and target domains are similar (e.g., both are natural images)
> - Training from scratch would take too long or require too much compute
> 
> Two strategies: **feature extraction** (freeze base, train only the head) for very small datasets; **fine-tuning** (unfreeze top layers with very low LR) when you have more data.

---

**Q5: What's the difference between `model.compile(loss='sparse_categorical_crossentropy')` and `categorical_crossentropy`?**

> Both compute cross-entropy loss for multi-class classification:
> - **`sparse_categorical_crossentropy`**: expects integer labels [0, 3, 1, 2, ...] — more memory efficient
> - **`categorical_crossentropy`**: expects one-hot encoded labels [[1,0,0], [0,0,0,1], ...]
> 
> The math is identical; the difference is only in label format. Use `sparse` when your labels are integers (most common), `categorical` when they're one-hot.

---

**Q6: How would you implement early stopping in Keras?**

> ```python
> early_stop = keras.callbacks.EarlyStopping(
>     monitor='val_loss',        # watch validation loss
>     patience=10,               # stop if no improvement for 10 epochs
>     restore_best_weights=True, # revert to best checkpoint on stop
>     min_delta=1e-4,            # minimum change to qualify as improvement
>     mode='min'                 # 'min' for loss, 'max' for accuracy/AUC
> )
> model.fit(..., callbacks=[early_stop])
> ```
> `restore_best_weights=True` is critical — without it, you get the weights from the last epoch (which might be worse than earlier epochs).

## 12. Resources

### Official
- **Keras API Reference:** https://keras.io/api/
- **Keras Examples:** https://keras.io/examples/
- **Keras Applications (pre-trained models):** https://keras.io/api/applications/
- **Keras Tuner:** https://keras.io/keras_tuner/

### Books
- **Deep Learning with Python** (by François Chollet, Keras creator): https://www.manning.com/books/deep-learning-with-python-second-edition
- **Hands-On Machine Learning** (Aurélien Géron): https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/

### Papers
- **Dropout (Srivastava et al.):** https://www.jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf
- **Batch Normalization (Ioffe & Szegedy):** https://arxiv.org/abs/1502.03167
- **Focal Loss for Dense Object Detection:** https://arxiv.org/abs/1708.02002
- **EfficientNet:** https://arxiv.org/abs/1905.11946

### Courses
- **Coursera Deep Learning Specialization:** https://www.coursera.org/specializations/deep-learning
- **fast.ai (practical DL for coders):** https://www.fast.ai/

## 13. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **Layer types** | Dense, Conv1D/2D, LSTM/GRU, Embedding, BatchNorm, Dropout, Attention |
| **Activation functions** | ReLU for hidden layers; Sigmoid for binary output; Softmax for multi-class; GELU for Transformers |
| **Regularization** | Dropout + L2 + BatchNorm + EarlyStopping work together |
| **LR schedules** | CosineDecay, ExponentialDecay, WarmupDecay — always better than fixed LR |
| **Transfer learning** | Freeze base → train head → unfreeze top + fine-tune at 100x lower LR |
| **Custom layers** | Subclass `keras.layers.Layer`, implement `call()` |
| **Custom losses** | Subclass `keras.losses.Loss`, implement `call(y_true, y_pred)` |
| **Keras Tuner** | Automated hyperparameter search with `RandomSearch`, `BayesianOptimization` |
| **Text CNNs** | Embedding → Conv1D → GlobalMaxPool → Dense for sequence classification |

### What's Next

**JAX** — the final deep learning framework in this phase. JAX is Google's NumPy-like library with:
- Automatic differentiation (`jax.grad`)
- JIT compilation (`jax.jit`) — makes Python code as fast as C++
- Vectorization (`jax.vmap`) — apply a function across a batch without explicit loops
- Parallelism (`jax.pmap`) — run across multiple devices/GPUs

JAX is the backbone of cutting-edge research libraries like **Flax**, **Haiku**, **Optax**, and **Equinox**.